# 🤖 Scikit-Learn: Zero to Hero
## Complete ML Notebook — Beginner → Production

**What you will learn:**
- Data loading, splitting, and stratification
- Feature preprocessing (scaling, encoding, imputation)
- Building sklearn Pipelines (the professional way)
- Multiple classifiers: Logistic Regression, Random Forest, SVM, Gradient Boosting
- Proper evaluation: Confusion Matrix, ROC-AUC, Classification Report
- Cross-validation & Hyperparameter Tuning (GridSearchCV, RandomizedSearchCV)
- Feature Importance & Model Explainability
- **Capstone: Titanic Survival Predictor** — resume-ready ML pipeline

> **Iris** is used in Section 1–3 for clean, numeric-only concept building.  
> **Titanic** is used in Section 4+ for real-world, messy, mixed-type ML.


## 📦 Section 1: Data Splitting — The Foundation

We use Iris (clean, numeric-only) to learn splitting concepts without distraction.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

iris = load_iris()
X, y = iris.data, iris.target

print(f'Dataset shape: {X.shape}')
print(f'Classes: {iris.target_names}')
print(f'Features: {iris.feature_names}')


### 1.1 Simple Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')


### 1.2 Stratified Split — Preserving Class Balance

> **Why it matters:** Without `stratify=y`, your split might accidentally skew class ratios.
> Stratification guarantees each split mirrors the original class distribution.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, split_y, title in zip(axes, [y_train, y_test], ['Train set', 'Test set']):
    counts = np.bincount(split_y)
    ax.bar(iris.target_names, counts, color=['steelblue', 'coral', 'seagreen'])
    ax.set_title(title)
    ax.set_ylabel('Count')
plt.suptitle('Class Distribution After Stratified Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## ⚙️ Section 2: Feature Preprocessing

Preprocessing transforms raw data into a format models can learn from effectively.


### 2.1 StandardScaler — Zero Mean, Unit Variance

> **When to use:** Most algorithms (Logistic Regression, SVM, KNN) are scale-sensitive.
> **Golden rule:** Fit ONLY on train data, then transform both train and test.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform on train: learns mean/std FROM training data only
X_train_scaled = scaler.fit_transform(X_train)
# transform on test: applies the SAME stats (no leakage)
X_test_scaled  = scaler.transform(X_test)

print(f'Before — mean: {X_train[:, 0].mean():.3f}, std: {X_train[:, 0].std():.3f}')
print(f'After  — mean: {X_train_scaled[:, 0].mean():.3f}, std: {X_train_scaled[:, 0].std():.3f}')


### 2.2 MinMaxScaler — Scale to [0, 1] Range

> **When to use:** Neural networks, image data, or when bounded values are required.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

mm_scaler = MinMaxScaler()
X_train_mm = mm_scaler.fit_transform(X_train)
X_test_mm  = mm_scaler.transform(X_test)

print(f'Min: {X_train_mm.min():.3f}, Max: {X_train_mm.max():.3f}')


### 2.3 OneHotEncoder — Encoding Categorical Variables

> **When to use:** Categorical features with NO natural ordering (sex, city, embarkation port).


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

sample_cat = pd.DataFrame({'sex': ['male', 'female', 'male', 'female']})

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(sample_cat[['sex']])

print('Categories:', ohe.categories_)
print('Encoded:\n', encoded)


### 2.4 SimpleImputer — Handling Missing Values

> Real datasets always have NaN values. Imputation fills them intelligently.


In [ ]:
from sklearn.impute import SimpleImputer

# Numeric: fill NaN with column median (robust to outliers)
num_imputer = SimpleImputer(strategy='median')

# Categorical: fill NaN with most frequent value
cat_imputer = SimpleImputer(strategy='most_frequent')

print('Numeric strategy:   ', num_imputer.strategy)
print('Categorical strategy:', cat_imputer.strategy)


## 🔗 Section 3: Sklearn Pipelines — The Professional Way

A Pipeline chains preprocessing steps + model into **one object**.
This prevents data leakage and makes deployment clean.

```
Raw Data → Imputer → Scaler → Encoder → Model → Prediction
```


### 3.1 Simple Pipeline (Iris — numeric only)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

simple_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression(max_iter=1000))
])

simple_pipeline.fit(X_tr, y_tr)
print(f'Iris Pipeline Accuracy: {simple_pipeline.score(X_te, y_te):.4f}')


## 🚢 Section 4: Titanic Dataset — Real-World ML

The Iris dataset is clean and toy-like. Real ML always involves:
- Missing values
- Mixed types (numbers + strings)
- Irrelevant columns
- Class imbalance

The **Titanic dataset** has all of these. We build a complete, production-quality pipeline on it.


### 4.1 Load & Explore the Data


In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df = titanic.frame.copy()

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')


In [ ]:
df.head()


In [ ]:
print('Missing values per column:')
print(df.isnull().sum().sort_values(ascending=False))


In [ ]:
print('Data types:')
print(df.dtypes)
print('\nSurvival class balance:')
print(df['survived'].value_counts())


### 4.2 Feature Engineering

> Creating **new informative features** from existing ones — where ML becomes creative.


In [ ]:
df_eng = df.copy()

# 1. Family size
df_eng['family_size'] = df_eng['sibsp'] + df_eng['parch'] + 1

# 2. Is alone (solo travelers had different survival odds)
df_eng['is_alone'] = (df_eng['family_size'] == 1).astype(int)

# 3. Title extracted from name (social status proxy)
df_eng['title'] = df_eng['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Don': 'Rare', 'Lady': 'Rare',
    'Countess': 'Rare', 'Jonkheer': 'Rare', 'Sir': 'Rare',
    'Capt': 'Rare', 'Ms': 'Miss'
}
df_eng['title'] = df_eng['title'].map(title_map).fillna('Rare')

print('Title distribution:')
print(df_eng['title'].value_counts())
print('\nFamily size distribution:')
print(df_eng['family_size'].value_counts().sort_index())


### 4.3 Prepare Features and Target

> **Critical rule:** `X` must NEVER contain the target column (`survived`).
> Drop identifier columns, high-missing columns, and the target.


In [ ]:
DROP_COLS = ['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest', 'survived']

X = df_eng.drop(columns=DROP_COLS)
y = df_eng['survived'].astype(int)

# Safety check
assert 'survived' not in X.columns, 'ERROR: target leaked into features!'

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'Feature columns: {X.columns.tolist()}')
print(f'Survival rate: {y.mean():.2%}')


### 4.4 Stratified Train/Test Split


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')
print(f'Train survival rate: {y_train.mean():.2%}')
print(f'Test  survival rate: {y_test.mean():.2%}')


### 4.5 Build the Full Preprocessing Pipeline


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Identify column types AFTER feature engineering
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Numeric columns  ({len(num_cols)}): {num_cols}')
print(f'Categorical cols ({len(cat_cols)}): {cat_cols}')


In [ ]:
# Numeric pipeline: impute missing → scale
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical pipeline: impute missing → one-hot encode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
])

print('Preprocessor built successfully.')


## 🧠 Section 5: Training Multiple Models

We train four classifiers and compare them:
1. Logistic Regression — interpretable baseline
2. Random Forest — powerful ensemble
3. Gradient Boosting — often best on tabular data
4. SVM — robust with scaled features


### 5.1 Logistic Regression


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
print(f'Logistic Regression — Test Accuracy: {lr_pipeline.score(X_test, y_test):.4f}')


### 5.2 Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
print(f'Random Forest — Test Accuracy: {rf_pipeline.score(X_test, y_test):.4f}')


### 5.3 Gradient Boosting


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42))
])

gb_pipeline.fit(X_train, y_train)
print(f'Gradient Boosting — Test Accuracy: {gb_pipeline.score(X_test, y_test):.4f}')


### 5.4 Support Vector Machine (SVM)


In [ ]:
from sklearn.svm import SVC

svm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVC(probability=True, random_state=42))
])

svm_pipeline.fit(X_train, y_train)
print(f'SVM — Test Accuracy: {svm_pipeline.score(X_test, y_test):.4f}')


## 📊 Section 6: Proper Model Evaluation

> **Accuracy alone is misleading on imbalanced data.** Always look at the full picture.


### 6.1 Confusion Matrix + Classification Report


In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

y_pred  = rf_pipeline.predict(X_test)
y_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print('=' * 50)
print('RANDOM FOREST — EVALUATION REPORT')
print('=' * 50)
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Died', 'Survived'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Random Forest — Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 6.2 Cross-Validation — Is the Model Reliable?

> One test split might be lucky. Cross-validation gives a **statistical estimate** of real performance.


In [ ]:
from sklearn.model_selection import cross_val_score

pipelines = {
    'Logistic Regression': lr_pipeline,
    'Random Forest':       rf_pipeline,
    'Gradient Boosting':   gb_pipeline,
    'SVM':                 svm_pipeline,
}

print(f'{'Model':<25} {'CV Mean':>8} {'CV Std':>8}')
print('-' * 45)
cv_results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:<25} {scores.mean():>8.4f} {scores.std():>8.4f}')


### 6.3 ROC Curves — Visual Model Comparison


In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
for name, pipe in pipelines.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax)

ax.plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 🔧 Section 7: Hyperparameter Tuning

Finding the best model settings systematically — this separates real ML from guessing.


### 7.1 GridSearchCV — Exhaustive Search


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators':     [100, 200],
    'model__max_depth':        [None, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')
print(f'Test ROC-AUC:    {roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1]):.4f}')


### 7.2 RandomizedSearchCV — Faster for Large Search Spaces


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'model__n_estimators':      randint(100, 500),
    'model__max_depth':         [None, 5, 10, 15, 20],
    'model__min_samples_split': randint(2, 20),
    'model__min_samples_leaf':  randint(1, 10),
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_dist,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f'Best Parameters: {random_search.best_params_}')
print(f'Best CV ROC-AUC: {random_search.best_score_:.4f}')


## 🔍 Section 8: Feature Importance & Explainability

Understanding WHICH features drive predictions is crucial for trust and model improvement.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

best_rf = grid_search.best_estimator_
preprocessor_fitted = best_rf.named_steps['preprocessor']

# Get encoded categorical feature names
ohe_feature_names = (
    preprocessor_fitted
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(cat_cols)
    .tolist()
)
all_feature_names = num_cols + ohe_feature_names

importances = best_rf.named_steps['model'].feature_importances_
feat_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(feat_df['feature'][::-1], feat_df['importance'][::-1], color='steelblue')
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Feature Importances — Tuned Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 💾 Section 9: Saving & Loading the Model

A real ML project is not complete until you can save and reload the model for production use.


In [ ]:
import joblib
import os

model_path = 'titanic_model.joblib'
joblib.dump(grid_search.best_estimator_, model_path)
print(f'Model saved to: {model_path}')
print(f'File size: {os.path.getsize(model_path) / 1024:.1f} KB')


In [ ]:
# Reload and use (simulates production inference)
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]
pred = loaded_model.predict(sample)[0]
prob = loaded_model.predict_proba(sample)[0][1]

print(f'Passenger features:')
print(sample.to_dict('records')[0])
print(f"\nPrediction: {'Survived' if pred == 1 else 'Did not survive'}")
print(f'Survival probability: {prob:.2%}')


## 🏆 Section 10: Capstone Project Summary

### What You Built

A **full production-grade ML pipeline** for Titanic survival prediction:

| Step | What you did |
|------|--------------|
| Data Loading | Fetched Titanic from OpenML |
| Exploration | Analyzed missing values, types, class balance |
| Feature Engineering | Created `family_size`, `is_alone`, `title` |
| Preprocessing | Numeric + categorical pipelines via ColumnTransformer |
| Modelling | Trained 4 classifiers: LR, RF, GBM, SVM |
| Evaluation | Confusion matrix, ROC-AUC, CV, ROC curves |
| Tuning | GridSearchCV + RandomizedSearchCV |
| Explainability | Feature importance chart |
| Deployment | Saved and reloaded with joblib |

### Resume Line
> *Built an end-to-end ML pipeline using scikit-learn on the Titanic dataset. Applied feature engineering, stratified splitting, ColumnTransformer pipelines, cross-validation, GridSearchCV hyperparameter tuning, and ROC-AUC evaluation. Compared Logistic Regression, Random Forest, Gradient Boosting, and SVM; saved final model with joblib.*

### Key sklearn Concepts Mastered
- `train_test_split` with `stratify`
- `StandardScaler`, `MinMaxScaler`, `OneHotEncoder`, `SimpleImputer`
- `Pipeline`, `ColumnTransformer`
- `LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier`, `SVC`
- `cross_val_score`, `GridSearchCV`, `RandomizedSearchCV`
- `confusion_matrix`, `classification_report`, `roc_auc_score`, `RocCurveDisplay`
- Feature importance extraction
- `joblib.dump` / `joblib.load`
